# RoBERTA on agnews
### LLM data generation: **"Llama-2-7b-chat-hf"**

https://huggingface.co/FacebookAI/roberta-base  
https://huggingface.co/FacebookAI/roberta-large

- total number of train samples: 500
- total number of test&nbsp; samples: 500
- classification model: "roberta-large"
- dataset: agnews

Generation methods:
- **A** generic augmentation
- **B** targeted augmentation
- **C** unsupervised context augmentation
- **D** real data with labels generated in zero-shot settings

TESTS:
1. 500 real
2. 250 real + 250 synthetic (A, B, C, D) [50% synthetic]
3. 50 real + 450 synthetic (A, B, C, D) [90% synthetic]
4. 500 synthetic (A, B, C, D) [100% synthetic]

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import json
from sklearn.model_selection import train_test_split

# CHANGE WORKING DIRECTORY TO ROOT
current_dir = os.path.basename(os.getcwd())
if current_dir == "src":
    os.chdir("..") # Move up by 1
elif os.path.basename(os.getcwd()) == "bai-thesis-nlp":  
    pass # If already at root, stay there
else:
    os.chdir("../..") # Move up by 2 otherwise
     
from src._utils._helpers import get_generated_examples_df
from src._utils._data_analysis_helpers import plot_micromacrof1
from src._utils._run_multiclassRoBERTA import main_multiclassRoBERTA

In [ ]:
LLM_NAME = "Llama-2-7b-chat-hf"
LLM_NAME_ZEROSHOT = "Llama-2-7b-hf"
DATASET_NAME = "agnews"

FOLDER_DIR = "src/"+DATASET_NAME+"/experiments/RoBERTA_500samples/data_"+LLM_NAME
os.makedirs(FOLDER_DIR, exist_ok=True)
LOG_DIR = os.path.join(FOLDER_DIR, "RoBERTA_log.json")

### LOAD DATA ###
# real data
real_train_df = pd.read_csv("real_data/train/"+DATASET_NAME+"trainAll.csv").rename(columns={"2": "text", "3": "label"})
real_train_df.drop(columns=["0", "1"], inplace=True)

# Take 500 samples for dev set
real_train_df, dev_df = train_test_split(real_train_df, test_size=500, random_state=42, stratify=real_train_df["label"])
# Store the dev set into a CSV file
dev_df.to_csv(os.path.join(FOLDER_DIR, "dev_set.csv"), index=False)

# synthetic data (generic, targeted and unsupervised context)
SYNTHETIC_DATA_DIR = "synthetic_data/datasets/"+LLM_NAME+"/"
syn_generic_df, _ = get_generated_examples_df(SYNTHETIC_DATA_DIR+DATASET_NAME+"_baseline_500.json")
syn_targeted_df, _ = get_generated_examples_df(SYNTHETIC_DATA_DIR+DATASET_NAME+"_targeted+tags_500.json")
syn_targeted_df = syn_targeted_df.drop(columns=["phenomena"], inplace=False)
syn_unsupContext_df, _ = get_generated_examples_df(SYNTHETIC_DATA_DIR+DATASET_NAME+"_unsupervisedContext_500.json")
syn_unsupContext_df = syn_unsupContext_df.drop(columns=["context_examples"])

In [ ]:
# zeroshot data (real data + zeroshot generated labels)
data = "./src/"+DATASET_NAME+"/experiments/fewshotCasualLM_500samples/"+LLM_NAME_ZEROSHOT+"_zeroshot_TRAIN.csv"
zeroshot_df = pd.read_csv(data)

display(zeroshot_df.head())
acc = sum(zeroshot_df['label'] == zeroshot_df['predicted_label']) / len(zeroshot_df)
print(f"Accuracy {acc:.3f} | number of samples: {zeroshot_df.shape[0]}"),

# keep only the labels that are allowed
labels = zeroshot_df['label'].unique()
zeroshot_df = zeroshot_df[zeroshot_df["predicted_label"].isin(labels)].reset_index(drop=True)
new_acc = sum(zeroshot_df['label'] == zeroshot_df['predicted_label']) / len(zeroshot_df)

# set the predicted label as the real label for training
zeroshot_df['label'] = zeroshot_df["predicted_label"]
zeroshot_df = zeroshot_df.drop(columns=["predicted_label"])

print("Filter out labels that are not in the real data. Results:")
print(f"Accuracy {new_acc:.3f} | number of samples: {zeroshot_df.shape[0]}")

In [ ]:
results = []

def get_results(details):
    """Just get useful results from the details"""
    res = {}
    res["generation_method"] = details["generation_method"]
    res["synthetic_ratio"] = details["synthetic_ratio"]
    for key, value in details["metrics_dev"].items():
        res[key] = value
    res["train_time"] = details["train_time"]
    res["eval_time"] = details["eval_time"]
    
    df = pd.DataFrame([res])
    display(df.round(3))
    return df

In [ ]:
base_config = {
    "real_df": real_train_df,
    # "synth_df": None/syn_generic_df..,
    "dev_df": dev_df,
    # "synth_ratio": 0.0/0.5..,
    "max_samples": 500,
    "epochs": 8,
    "batch_size": 16,
    "output_dir": FOLDER_DIR,
    "log_dir": LOG_DIR,
    # "generation_method": None/"generic"/"targeted",
    "save_model": False,
    "save_dataset": True,
}

## 1. 500 real

In [ ]:
config = base_config.copy()
config["synth_df"] = None
config["synth_ratio"] = 0.0
config["generation_method"] = None

train_details = main_multiclassRoBERTA(**config)
results.append(get_results(train_details))

## 2. 250 real + 250 synthetic

A. Generic augmentation

In [ ]:
config = base_config.copy()
config["synth_df"] = syn_generic_df # Generic Augmentation
config["synth_ratio"] = 0.5 # 50% of the data is synthetic
config["generation_method"] = "generic"

train_details = main_multiclassRoBERTA(**config)
results.append(get_results(train_details))

B. Targeted augmentation

In [ ]:
config = base_config.copy()
config["synth_df"] = syn_targeted_df # Targeted Augmentation
config["synth_ratio"] = 0.5 # 50% of the data is synthetic
config["generation_method"] = "targeted"

train_details = main_multiclassRoBERTA(**config)
results.append(get_results(train_details))

C. Unsupervised Context augmentation

In [ ]:
config = base_config.copy()
config["synth_df"] = syn_unsupContext_df # Unsupervised Context Augmentation
config["synth_ratio"] = 0.5 # 50% of the data is synthetic
config["generation_method"] = "unsupContext"

train_details = main_multiclassRoBERTA(**config)
results.append(get_results(train_details))

D. Zeroshot labels

In [ ]:
config = base_config.copy()
config["synth_df"] = zeroshot_df # real data with labels predicted in zero-shot setting
config["synth_ratio"] = 0.5 # 50% of the data is synthetic
config["generation_method"] = "zeroshotLabels"

train_details = main_multiclassRoBERTA(**config)
results.append(get_results(train_details))

## 3. 50 real + 450 synthetic
A. Generic augmentation

In [ ]:
config = base_config.copy()
config["synth_df"] = syn_generic_df # Generic Augmentation
config["synth_ratio"] = 0.9 # 90% of the data is synthetic
config["generation_method"] = "generic"

train_details = main_multiclassRoBERTA(**config)
results.append(get_results(train_details))

B. Targeted augmentation

In [ ]:
config = base_config.copy()
config["synth_df"] = syn_targeted_df # Targeted Augmentation
config["synth_ratio"] = 0.9 # 90% of the data is synthetic
config["generation_method"] = "targeted"

train_details = main_multiclassRoBERTA(**config)
results.append(get_results(train_details))

C. Unsupervised Context augmentation

In [ ]:
config = base_config.copy()
config["synth_df"] = syn_unsupContext_df # Unsupervised Context Augmentation
config["synth_ratio"] = 0.9 # 90% of the data is synthetic
config["generation_method"] = "unsupContext"

train_details = main_multiclassRoBERTA(**config)
results.append(get_results(train_details))

D. Zeroshot labels

In [ ]:
config = base_config.copy()
config["synth_df"] = zeroshot_df # real data with labels predicted in zero-shot setting
config["synth_ratio"] = 0.9 # 90% of the data is synthetic
config["generation_method"] = "zeroshotLabels"

train_details = main_multiclassRoBERTA(**config)
results.append(get_results(train_details))

## 4. 500 synthetic
A. Generic augmentation

In [ ]:
config = base_config.copy()
config["real_df"] = None
config["synth_df"] = syn_generic_df # Generic Augmentation
config["synth_ratio"] = 1.0 # 100% of the data is synthetic
config["generation_method"] = "generic"
config["save_dataset"] = False

train_details = main_multiclassRoBERTA(**config)
results.append(get_results(train_details))

B. Targeted augmentation

In [ ]:
config = base_config.copy()
config["real_df"] = None
config["synth_df"] = syn_targeted_df # Targeted Augmentation
config["synth_ratio"] = 1.0 # 100% of the data is synthetic
config["generation_method"] = "targeted"
config["save_dataset"] = False

train_details = main_multiclassRoBERTA(**config)
results.append(get_results(train_details))

C. Unsupervised Context augmentation

In [ ]:
config = base_config.copy()
config["real_df"] = None
config["synth_df"] = syn_unsupContext_df # Unsupervised Context Augmentation
config["synth_ratio"] = 1.0 # 100% of the data is synthetic
config["generation_method"] = "unsupContext"
config["save_dataset"] = False

train_details = main_multiclassRoBERTA(**config)
results.append(get_results(train_details))

D. Zeroshot labels

In [ ]:
config = base_config.copy()
config["real_df"] = None
config["synth_df"] = zeroshot_df # real data with labels predicted in zero-shot setting
config["synth_ratio"] = 1.0 # 100% of the data is synthetic
config["generation_method"] = "zeroshotLabels"
config["save_dataset"] = False

train_details = main_multiclassRoBERTA(**config)
results.append(get_results(train_details))


---
# Results

In [ ]:
df_results = pd.concat(results).reset_index(drop=True)
df_results.to_csv(os.path.join(FOLDER_DIR, "RoBERTA_results_dev.csv"), index=False)
display(df_results.round(3))

In [ ]:
df_results = pd.read_csv(os.path.join(FOLDER_DIR, "RoBERTA_results_dev.csv"))

plot_micromacrof1(df=df_results, x_col="synthetic_ratio", hue_col="generation_method")

In [ ]:
import pandas as pd

# Define a function to extract fields
def split_method(m):
    # Check for special case
    if 'zeroshotLabel' in m:
        return pd.Series([m, 'zeroshotLabel'])
    elif 'generic' in m:
        config = m.replace('generic', 'synthetic')
        return pd.Series([config, 'generic'])
    elif 'targeted' in m:
        config = m.replace('targeted', 'synthetic')
        return pd.Series([config, 'targeted'])
    elif 'unsupContext' in m:
        config = m.replace('unsupContext', 'synthetic')
        return pd.Series([config, 'unsupContext'])
    elif 'real' in m and '_' not in m:
        return pd.Series([m, 'real'])
    else:
        return pd.Series([m, 'unknown'])

# Apply the function
df_results[['configuration', 'generation_method']] = df_results['method'].apply(split_method)

display(df_results)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
from matplotlib.lines import Line2D

# Set style
sns.set_style("darkgrid")


# Melt the data to long format
df_long = df_results.melt(
    id_vars=["configuration", "generation_method"],
    value_vars=["f1_micro", "f1_macro", "accuracy"],
    var_name="metric",
    value_name="score"
)

# Set the figure
plt.figure(figsize=(14, 6))

# Create a grouped barplot
barplot = sns.barplot(
    data=df_long,
    x="metric",
    y="score",
    hue="configuration",
    errorbar=None,
    palette="pastel"
)

# Compute positions of bars to align dots
metric_order = df_long['metric'].unique()
config_order = df_long['configuration'].unique()
group_width = 0.8
n_configs = len(config_order)
bar_width = group_width / n_configs

# Compute x positions for bars
x_positions = {}
for i, metric in enumerate(metric_order):
    for j, config in enumerate(config_order):
        x = i - group_width/2 + j * bar_width + bar_width / 2
        x_positions[(metric, config)] = x

# Define marker styles for generation methods
marker_styles = {
    # 'real': ('d', 'black'),
    'generic': ('s', 'firebrick'),
    'targeted': ('o', 'mediumblue'),
    'unsupContext': ('X', 'forestgreen'),
    # 'zeroshotLabel': ('^', 'orange')
}


# Only plot dots for selected generation methods
excluded_methods = ['real', 'zeroshotLabel']

for _, row in df_long.iterrows():
    if row['generation_method'] in excluded_methods:
        continue  # skip plotting for real and zeroshotLabel

    x = x_positions[(row['metric'], row['configuration'])]
    marker, edge_color = marker_styles[row['generation_method']]
    plt.scatter(
        x, row['score'],
        facecolors='none',
        edgecolors=edge_color,
        marker=marker,
        s=90,
        linewidth=1.6,
        zorder=5
    )
    tick_width = 0.06
    plt.plot([x - tick_width, x + tick_width], [row['score'], row['score']], color=edge_color, lw=1.7, zorder=4)


# Custom legend (excluding real and zeroshotLabel)
scatter_handles = [
    Line2D([0], [0], marker=m, color=c, label=label, markerfacecolor='none',
           markeredgewidth=1.4, markersize=8, linestyle='None')
    for label, (m, c) in marker_styles.items() if label not in excluded_methods
]


# Plot customization
plt.title("Metric Scores by Configuration (Bars) and Generation Method (Markers)")
plt.ylabel("Score")
plt.xlabel("Metric")
plt.xticks(ticks=range(len(metric_order)), labels=metric_order)

# Legends
bar_handles, bar_labels = barplot.get_legend_handles_labels()
legend1 = plt.legend(bar_handles, bar_labels, title='Configuration (Bars)', loc='upper left', bbox_to_anchor=(1, 1))
plt.legend(scatter_handles, marker_styles.keys(), title='Generation Method (Markers)', loc='upper left', bbox_to_anchor=(1, 0.55))
plt.gca().add_artist(legend1)

plt.tight_layout()
plt.show()


---

In [ ]:
import pandas as pd
import re

def split_method(m):
    # Special case
    if 'zeroshotLabel' in m:
        return pd.Series([m, 'zeroshotLabel', None])

    # Extract numbers and labels
    match = re.match(r"(\d+)real(?:_(\d+)(generic|targeted|unsupContext))?", m)
    if match:
        real = int(match.group(1))
        if match.group(2) and match.group(3):
            synthetic = int(match.group(2))
            kind = match.group(3)
            config = f"{real}real_{synthetic}synthetic"
            ratio = synthetic / (real + synthetic)
            return pd.Series([config, kind, ratio])
        else:
            return pd.Series([m, 'real', 0.0])  # pure real: ratio 0

    # Pure synthetic cases
    match = re.match(r"(\d+)(generic|targeted|unsupContext)", m)
    if match:
        synthetic = int(match.group(1))
        kind = match.group(2)
        config = f"0real_{synthetic}synthetic"
        return pd.Series([config, kind, 1.0])  # pure synthetic: ratio 1

    # Unknown format
    return pd.Series([m, 'unknown', None])


In [ ]:
df_long = df_results.melt(
    id_vars=["method"],
    value_vars=["f1_micro", "f1_macro"],
    var_name="metric",
    value_name="score"
)

df_long[['config', 'kind', 'synthetic_ratio']] = df_long['method'].apply(split_method)
df_long

In [ ]:
sns.set_style("darkgrid")
sns.set_context("notebook")  # talk, paper

# Build relplot
g = sns.relplot(
    data=df_long,
    x="synthetic_ratio",
    y="score",
    col="metric",
    hue="kind",
    style="kind",
    kind="line",
    markers=True,
    dashes=False,
    height=6,
    aspect=1.2,
    facet_kws={'sharey': True}
)

# Adjust marker size and line width
for ax in g.axes.flat:
    for line in ax.lines:
        line.set_markersize(12)   # marker size
        line.set_linewidth(2)   # line width

g.set_titles(col_template="{col_name}")

# for ax in g.axes.flat:
#     ax.set_ylim(0, 1.05)

plt.show()